# Local versus global chemistry

This notebook is the slower companion to `reports/chemistry-local-to-global.md`.

The narrow question is the one that matters once the two chemistry atlases exist: **what does the Jacobian tell us exactly, and what only shows up after we integrate the trajectories for a long time?**

The repo now has two useful chemistry examples for that split:

- the **Brusselator**, where one exact Hopf curve divides a stable side from an oscillatory side
- the **Selkov** model, where two exact Hopf curves create a finite oscillatory band


## Local theory versus global measurement

Local linearization is about the fixed point itself.
It tells us whether small perturbations decay or grow nearby.

Global measurement asks a different question: what does the long-time orbit actually do on the finite grid used in this repo?

That means we care about things like:

- whether a visible limit cycle appears at all
- how large that cycle becomes
- how long one turn takes
- and how sharply those quantities change near a Hopf boundary

Those are not Jacobian-only quantities. They need trajectory integration.


In [ ]:
from phaseportraitlab.brusselator_atlas import default_brusselator_parameter_atlas
from phaseportraitlab.brusselator_sweep import brusselator_hopf_threshold
from phaseportraitlab.selkov_atlas import default_selkov_parameter_atlas, selkov_hopf_band

br_cells = default_brusselator_parameter_atlas()
sel_cells = default_selkov_parameter_atlas()

def nearest(cells, **targets):
    return min(cells, key=lambda cell: sum(abs(getattr(cell, key) - value) for key, value in targets.items()))

br_anchors = [
    nearest(br_cells, a=1.0, b=1.6),
    nearest(br_cells, a=1.0, b=2.1),
    nearest(br_cells, a=1.0, b=2.4),
]

sel_anchors = [
    nearest(sel_cells, a=0.08, b=0.25),
    nearest(sel_cells, a=0.08, b=0.35),
    nearest(sel_cells, a=0.08, b=0.60),
    nearest(sel_cells, a=0.08, b=0.95),
]

brusselator_hopf_threshold(1.0), selkov_hopf_band(0.08)


## Brusselator: onset is local, scale is global

For the Brusselator

$$\dot x = A + x^2 y - (B + 1)x, \qquad \dot y = Bx - x^2 y,$$

the positive fixed point is $(x^*, y^*) = (A, B/A)$ and the exact Hopf threshold is

$$B = 1 + A^2.$$

That tells us exactly where the fixed point stops attracting.
It does **not** tell us the size of the emerging cycle on the finite scan.


In [ ]:
[(round(cell.a, 2), round(cell.b, 2), round(cell.trace, 3), cell.classification, round(cell.x_amplitude, 2), None if cell.period is None else round(cell.period, 2)) for cell in br_anchors]


A useful reading from that table:

- below threshold, the fixed point attracts and the measured amplitude is zero
- just above threshold, the trace is only slightly positive but the cycle is already real
- farther above threshold, the cycle amplitude is much larger even though the local classification stays the same

That is the first place where local theory stops being enough.


## Selkov: bounded instability changes the story

For the Selkov model

$$\dot x = -x + a y + x^2 y, \qquad \dot y = b - a y - x^2 y,$$

the Jacobian gives a finite Hopf band when $a < 1/8$.
For `a = 0.08`, the exact unstable window is a bounded interval in `b`, not a one-sided half-plane.

That already makes the local geometry richer than the Brusselator.
But we still need trajectory measurements to see where the cycle is tiny, where it is strong, and where attraction returns.


In [ ]:
[(round(cell.a, 3), round(cell.b, 2), round(cell.trace, 3), cell.classification, round(cell.x_amplitude, 2), None if cell.period is None else round(cell.period, 2)) for cell in sel_anchors]


The middle two rows are the key ones.
Near the lower edge of the band, the cycle is present but still small.
Deeper inside the band, the same local label `unstable spiral` hides a much larger oscillation.
Above the upper edge, the fixed point becomes attracting again and the measured cycle disappears.


## What this repo can now say honestly

With the two chemistry models side by side, `phase-portrait-lab` can now make a more careful claim than a single Hopf example allows:

1. the Jacobian gives exact local onset information for these models
2. finite-time integration shows what that onset turns into on a visible parameter grid
3. the global parameter geography is model-specific even when the local word "Hopf" appears in both stories

That is the whole point of the local/global split here.


## Caveats, problems, and next checks

Caveats:

- amplitude and period here come from finite RK4 runs, not a full continuation method
- near-threshold cells are the most sensitive to run length and transient trimming
- a local instability flag does not prove the rest of the global geometry

Good next problems:

1. tighten the near-threshold Brusselator measurements with two integration horizons and compare drift
2. measure where the Selkov period is largest relative to distance from the band center
3. add one chemistry example whose global geometry is not just another Hopf-band variant

Files worth opening after this notebook:

- `assets/brusselator-parameter-atlas.svg`
- `assets/selkov-parameter-atlas.svg`
- `reports/brusselator-parameter-atlas.md`
- `reports/selkov-parameter-atlas.md`
- `reports/chemistry-local-to-global.md`
